In [ ]:
!pip install -q -U transformers accelerate datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
!pip3 install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")

dev   = ds["train"].select(range(0, 10))    # train 1-10
demos = ds["train"].select(range(10, 14))   # train 11-14
test  = ds["test"].select(range(0, 30))     # test 1-30


In [ ]:
print(len(dev), len(demos), len(test))

10 4 30


In [ ]:
print(dev[0]["question"])
print(dev[0]["answer"])

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [ ]:
SYSTEM = "You are a helpful assistant. Solve the math problem step by step. End with '#### <number>'."

def zero_shot(question):
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]

In [ ]:
 def four_shot(question):
    msgs = [{"role": "system", "content": SYSTEM}]
    for d in demos:                       # train[10:14]
        msgs.append({"role": "user", "content": d["question"]})
        msgs.append({"role": "assistant", "content": d["answer"]})
    msgs.append({"role": "user", "content": question})
    return msgs

In [ ]:
import re

In [ ]:
def extract_number(text):
    m = re.search(r"####\s*\$?(-?[\d,]*\.?\d+)", text)
    s = m.group(1) if m else None
    if s is None:
        nums = re.findall(r"-?[\d,]*\.?\d+", text)
        if not nums:
            return None
        s = nums[-1]
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return None

In [ ]:
from collections import Counter

In [ ]:
STATS = {"calls": 0, "gen_tokens": 0}

def generate(messages, sample=False, n=1, max_new_tokens=512):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if sample:
        kw.update(do_sample=True, temperature=0.7, top_p=0.95, num_return_sequences=n)
    else:
        kw.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **kw)
    new = out[:, inputs.input_ids.shape[1]:]
    STATS["calls"] += 1
    STATS["gen_tokens"] += int((new != tokenizer.pad_token_id).sum())
    return tokenizer.batch_decode(new, skip_special_tokens=True)


def self_consistency(question,n=4):
  texts = generate(four_shot(question),sample=True,n=n)
  votes = [extract_number(t) for t in texts]
  votes = [v for v in votes if v is not None]
  return Counter(votes).most_common(1)[0][0] if votes else None

In [ ]:
def reflect (question,first_solution):

  return[{"role": "system", "content":SYSTEM},
         {"role":"user","content":
          f"Question: {question}\n\n"
          f"Here is a proposed solution :\n{first_solution}\n\n"
          "Review this solution carefully step by step. If you find a mistake, fix it."
          "End with the final answer as '#### <number>'."},]

def reflection_method(question):
  first = generate(four_shot(question))[0]
  revised = generate(reflect(question,first))[0]
  return extract_number(revised)

In [ ]:
!pip install -q -U transformers accelerate datasets

In [ ]:
def m_zero_shot(q):
  return extract_number(generate(zero_shot(q))[0])

def m_four_shot(q):
  return extract_number(generate(four_shot(q))[0])

def m_self_consistency(q,n=4):
  texts = generate(four_shot(q),sample=True,n=n)
  votes =[v for v in (extract_number(t) for t in texts) if v is not None]
  return Counter(votes).most_common(1)[0][0] if votes else None

def m_reflection(q):
  first = generate(four_shot(q))[0]
  revised = generate(reflect(q,first))[0]
  r=extract_number(revised)
  return r if r is not None else extract_number(first)


In [ ]:

dev_questions  = [ex["question"] for ex in dev]
dev_gold       = [float(ex["answer"].split("####")[-1].strip().replace(",", "")) for ex in dev]
test_questions = [ex["question"] for ex in test]
test_gold      = [float(ex["answer"].split("####")[-1].strip().replace(",", "")) for ex in test]

In [ ]:
import re, time, torch

METHODS = {
    "zero_shot": m_zero_shot,
    "four_shot": m_four_shot,
    "self_consistency": m_self_consistency,
    "reflection": m_reflection,
}

def run(method, questions):
    preds = []
    for i, q in enumerate(questions):
        torch.manual_seed(42)
        preds.append(METHODS[method](q))
        print(f"  {method}: {i+1}/{len(questions)}", end="\r")
    return preds

def accuracy(preds,golds):
  correct = sum(p is not None and abs(p-g) < 1e-6 for p ,g in zip(preds,golds))
  return correct , len(golds)

def evaluate(questions, golds):
    results = {}
    for name in METHODS:
        STATS.update(calls=0, gen_tokens=0)
        t0 = time.time()
        preds = run(name, questions)
        elapsed = time.time() - t0
        c, n = accuracy(preds, golds)
        results[name] = dict(correct=c, n=n, time=elapsed,
                             calls=STATS["calls"], tokens=STATS["gen_tokens"], preds=preds)
        print(f"{name:18s} {c}/{n} = {c/n:.1%} | calls={STATS['calls']} | "
              f"generated_tokens={STATS['gen_tokens']} | time={elapsed:.0f}s")
    return results

In [ ]:
dev_results = evaluate(dev_questions, dev_gold)

zero_shot          5/10 = 50.0% | calls=10 | generated_tokens=2537 | time=106s
four_shot          3/10 = 30.0% | calls=10 | generated_tokens=1537 | time=58s
self_consistency   6/10 = 60.0% | calls=10 | generated_tokens=7644 | time=97s
reflection         6/10 = 60.0% | calls=20 | generated_tokens=3465 | time=121s


In [ ]:
preds = dev_results["four_shot"]["preds"]
for q, p, g in zip(dev_questions, preds, dev_gold):
    if p is None or abs(p - g) > 1e-6:
        print("Question:", q, "\nPrediction:", p, "| Correct Value:", g, "\n")

Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? 
Prediction: 50.0 | Correct Value: 10.0 

Question: Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read? 
Prediction: 36.0 | Correct Value: 42.0 

Question: James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year? 
Prediction: 416.0 | Correct Value: 624.0 

Question: Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden? 
Prediction: 32.0 | Correct Value: 35.0 

Question: Ken created a care package to send to his brother, who was away 

In [ ]:
for i, q in enumerate(dev_questions):
    out = generate(four_shot(q))[0]
    p = extract_number(out)
    if p is None or abs(p - dev_gold[i]) > 1e-6:
        print(f"===== Soru {i} | tahmin: {p} | doğru: {dev_gold[i]}")
        print(out)
        print()

===== Soru 1 | tahmin: 50.0 | doğru: 10.0
To calculate Weng's earnings, we need to convert the time from minutes to hours. There are 60 minutes in one hour, so 50 minutes is equal to 50/60 hours.

Earnings = (Time in hours) × (Hourly wage)
Earnings = (50/60) × $12

Now let's do the calculation:

Earnings = (50/60) × $12
Earnings = (5/6) × $12
Earnings = $<<5/6*12=10>>10

Weng earned $10 for the 50 minutes of babysitting.

===== Soru 3 | tahmin: 36.0 | doğru: 42.0
Yesterday, Julie read 12 pages out of a total of 120 pages, which means she still needs to read 120 - 12 = <<120-12=108>>108 pages.
Today, she read twice as many pages as yesterday, which means she read 12 x 2 = <<12*2=24>>24 pages.
After today, she has read a total of 12 + 24 = <<12+24=36>>36 pages.
So, she still needs to read 108 - 36 = <<108-36=72>>72 pages tomorrow.
She wants to read half of these remaining pages, which means she should read 72 / 2 = <<72/2=36>>36 pages tomorrow.
#### 36

===== Soru 4 | tahmin: 416.0 | doğ

In [ ]:
for i, q in enumerate(dev_questions):
    out = generate(zero_shot(q))[0]
    p = extract_number(out)
    print(f"[{i}] pred={p} gold={dev_gold[i]}")
    print(out[-300:], "\n")

[0] pred=96.0 gold=72.0
f clips sold in April to the number of clips sold in May to get the total number of clips sold in both months:
\[ \text{Total clips sold} = \text{Clips sold in April} + \text{Clips sold in May} = 48 + 24 = 72 \]

Therefore, the total number of clips Natalia sold in April and May is:
\[ \boxed{72} \] 

[1] pred=600.0 gold=10.0
To calculate how much Weng earned for babysitting, you can multiply her hourly rate by the number of hours she worked.

Hourly rate: $12
Hours worked: 50

$12/hour * 50 hours = $600

So, Weng earned $600 yesterday from babysitting. 

[2] pred=25.0 gold=5.0
ount needed} = 50 + 45 = 95
     \]

4. **Determine how much more money Betty needs:**
   - Subtract the amount she currently has from the total amount needed.
     \[
     \text{Additional amount needed} = 95 - 50 = 45
     \]

Thus, Betty needs an additional $\boxed{45}$ dollars to buy the wallet. 

[3] pred=60.0 gold=42.0
 number of pages left to read is \(120 - 36 = 84\) pages.

Final

In [ ]:
SYSTEM = (
    "You are a helpful assistant. Solve the math problem step by step. "
    "Your final line must be exactly '#### <number>', where <number> is only the final numeric answer "
    "(no units, no $ sign, no extra text)."
)

def extract_number(text):
    # 1) #### kalıbı
    m = re.search(r"####\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        # 2) "answer is / earned / total ..." gibi son cümlede $ ile gelen sayı
        last_line = [l for l in text.strip().split("\n") if l.strip()][-1]
        dollars = re.findall(r"\$\s*(-?[\d,]*\.?\d+)", last_line)
        nums = dollars or re.findall(r"-?[\d,]*\.?\d+", last_line)
        if not nums:
            return None
        s = nums[0] if dollars else nums[-1]
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return None

In [ ]:
dev_results = evaluate(dev_questions, dev_gold)

zero_shot          1/10 = 10.0% | calls=10 | generated_tokens=2647 | time=101s
four_shot          3/10 = 30.0% | calls=10 | generated_tokens=1540 | time=56s
self_consistency   4/10 = 40.0% | calls=10 | generated_tokens=7668 | time=97s
reflection         5/10 = 50.0% | calls=20 | generated_tokens=3302 | time=117s


In [ ]:
def extract_number(text):
    # 1) '####' pattern (take the last occurrence)
    ms = re.findall(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if not ms:
        # 2) \boxed{...} pattern (take the last occurrence)
        ms = re.findall(r"\\boxed\{\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if ms:
        s = ms[-1]
    else:
        # 3) fallback: last number in the whole text (the original behaviour)
        nums = re.findall(r"-?[\d,]*\.?\d+", text)
        if not nums:
            return None
        s = nums[-1]
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return None

SYSTEM_A = "You are a helpful assistant. Solve the math problem step by step. End with '#### <number>'."
SYSTEM_B = ("You are a helpful assistant. Solve the math problem step by step. "
            "Your final line must be exactly '#### <number>', where <number> is only the final numeric answer "
            "(no units, no $ sign, no extra text).")

for name, s in [("A", SYSTEM_A), ("B", SYSTEM_B)]:
    SYSTEM = s
    print("SYSTEM", name)
    evaluate(dev_questions, dev_gold)

SYSTEM A
zero_shot          5/10 = 50.0% | calls=10 | generated_tokens=2537 | time=86s
four_shot          3/10 = 30.0% | calls=10 | generated_tokens=1537 | time=55s
self_consistency   6/10 = 60.0% | calls=10 | generated_tokens=7644 | time=97s
reflection         6/10 = 60.0% | calls=20 | generated_tokens=3465 | time=121s
SYSTEM B
zero_shot          1/10 = 10.0% | calls=10 | generated_tokens=2647 | time=90s
four_shot          3/10 = 30.0% | calls=10 | generated_tokens=1540 | time=56s
self_consistency   4/10 = 40.0% | calls=10 | generated_tokens=7668 | time=99s
reflection         5/10 = 50.0% | calls=20 | generated_tokens=3302 | time=118s


In [ ]:

SYSTEM = "You are a helpful assistant. Solve the math problem step by step. End with '#### <number>'."

def extract_number(text):
    ms = re.findall(r"####\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if not ms:
        ms = re.findall(r"\\boxed\{\s*\$?\s*(-?[\d,]*\.?\d+)", text)
    if ms:
        s = ms[-1]
    else:
        nums = re.findall(r"-?[\d,]*\.?\d+", text)
        if not nums:
            return None
        s = nums[-1]
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return None

print(SYSTEM)
print(extract_number("... \\boxed{72}"), extract_number("x #### $1,200"), extract_number("answer 5 then 10"))
# expected: 72.0 1200.0 10.0

You are a helpful assistant. Solve the math problem step by step. End with '#### <number>'.
72.0 1200.0 10.0


In [ ]:
import json, time, torch
from collections import Counter

STATS = {"calls": 0, "gen_tokens": 0}
RAW = []                                   # (method, question_index, kind, text)
CUR = {"method": None, "idx": None}

def generate(messages, sample=False, n=1, max_new_tokens=512):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if sample:
        kw.update(do_sample=True, temperature=0.7, top_p=0.95, num_return_sequences=n)
    else:
        kw.update(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, **kw)
    new = out[:, inputs.input_ids.shape[1]:]
    STATS["calls"] += n if sample else 1
    STATS["gen_tokens"] += int((new != tokenizer.pad_token_id).sum())
    texts = tokenizer.batch_decode(new, skip_special_tokens=True)
    for t in texts:
        RAW.append((CUR["method"], CUR["idx"], "sample" if sample else "greedy", t))
    return texts

def m_zero_shot(q):
    return extract_number(generate(zero_shot(q))[0])

def m_four_shot(q):
    return extract_number(generate(four_shot(q))[0])

def m_self_consistency(q, n=4):
    texts = generate(four_shot(q), sample=True, n=n)
    votes = [v for v in (extract_number(t) for t in texts) if v is not None]
    return Counter(votes).most_common(1)[0][0] if votes else None

def m_reflection(q):
    first = generate(four_shot(q))[0]
    revised = generate(reflect(q, first))[0]
    r = extract_number(revised)
    return r if r is not None else extract_number(first)

METHODS = {"zero_shot": m_zero_shot, "four_shot": m_four_shot,
           "self_consistency": m_self_consistency, "reflection": m_reflection}

def run(method, questions):
    preds = []
    for i, q in enumerate(questions):
        torch.manual_seed(42)
        CUR.update(method=method, idx=i)
        preds.append(METHODS[method](q))
        print(f"  {method}: {i+1}/{len(questions)}", end="\r")
    return preds

def accuracy(preds, golds):
    return sum(p is not None and abs(p - g) < 1e-6 for p, g in zip(preds, golds)), len(golds)

def evaluate(questions, golds):
    results = {}
    for name in METHODS:
        STATS.update(calls=0, gen_tokens=0)
        t0 = time.time()
        preds = run(name, questions)
        elapsed = time.time() - t0
        c, n = accuracy(preds, golds)
        results[name] = dict(correct=c, n=n, time=elapsed, calls=STATS["calls"],
                             tokens=STATS["gen_tokens"], preds=preds,
                             none_count=sum(p is None for p in preds))
        print(f"{name:18s} {c}/{n} = {c/n:.1%} | calls={STATS['calls']} | "
              f"generated_tokens={STATS['gen_tokens']} | None={results[name]['none_count']} | time={elapsed:.0f}s")
    return results

In [ ]:
RAW.clear()
final_results = evaluate(test_questions, test_gold)

zero_shot          11/30 = 36.7% | calls=30 | generated_tokens=9669 | None=0 | time=335s
four_shot          11/30 = 36.7% | calls=30 | generated_tokens=4906 | None=0 | time=176s
self_consistency   8/30 = 26.7% | calls=120 | generated_tokens=29800 | None=0 | time=378s
reflection         11/30 = 36.7% | calls=60 | generated_tokens=12304 | None=0 | time=429s


In [ ]:
print("| Method | Test accuracy | Total generated tokens | Model calls |")

for k, v in final_results.items():
    print(f"| {k} | {v['correct']}/{v['n']} ({v['correct']/v['n']:.1%}) | {v['tokens']:,} | {v['calls']} |")

| Method | Test accuracy | Total generated tokens | Model calls |
| zero_shot | 11/30 (36.7%) | 9,669 | 30 |
| four_shot | 11/30 (36.7%) | 4,906 | 30 |
| self_consistency | 8/30 (26.7%) | 29,800 | 120 |
| reflection | 11/30 (36.7%) | 12,304 | 60 |


In [ ]:
fs, rf, sc = (final_results[k]["preds"] for k in ["four_shot", "reflection", "self_consistency"])
ok = lambda p, g: p is not None and abs(p - g) < 1e-6

for i, g in enumerate(test_gold):
    a, b, c = ok(fs[i], g), ok(rf[i], g), ok(sc[i], g)
    tag = ""
    if not a and (b or c): tag = "<- extra inference HELPED"
    if a and not (b and c): tag = "<- extra inference HARMED"
    print(i, "4shot", a, "| refl", b, "| sc", c, tag)

0 4shot False | refl True | sc False <- extra inference HELPED
1 4shot True | refl False | sc False <- extra inference HARMED
2 4shot False | refl False | sc False 
3 4shot False | refl False | sc False 
4 4shot False | refl False | sc False 
5 4shot False | refl False | sc False 
6 4shot True | refl True | sc True 
7 4shot False | refl False | sc False 
8 4shot False | refl False | sc False 
9 4shot False | refl False | sc True <- extra inference HELPED
10 4shot True | refl True | sc True 
11 4shot True | refl True | sc True 
12 4shot False | refl False | sc False 
13 4shot False | refl False | sc False 
14 4shot False | refl False | sc True <- extra inference HELPED
15 4shot False | refl False | sc False 
16 4shot True | refl True | sc False <- extra inference HARMED
17 4shot False | refl True | sc False <- extra inference HELPED
18 4shot True | refl False | sc True <- extra inference HARMED
19 4shot False | refl False | sc False 
20 4shot False | refl False | sc False 
21 4shot Fals

In [ ]:
idx = 0
print("QUESTION:", test_questions[idx], "| GOLD:", test_gold[idx], "\n")
for m, i, kind, t in RAW:
    if i == idx and m in ("four_shot", "reflection", "self_consistency"):
        print(f"--- {m} ({kind}) ---\n{t}\n")

QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market? | GOLD: 18.0 

--- four_shot (greedy) ---
She eats 16 - 3 - 4 = <<16-3-4=19>>19 eggs daily.
She makes 19 * $2 = <<19*2=38>>38 selling the eggs at the farmers' market.
#### 38

--- self_consistency (sample) ---
She makes $16 - 3 - 4 = $<number>. She makes $<<16-3-4=17>>17 every day at the farmers' market.

--- self_consistency (sample) ---
She eats 16 eggs for breakfast which means Janet loses 16 eggs in one day. She also bakes muffins for her friends, meaning she loses an additional 16 muffins. Therefore, she is losing 16 + 16 = <<16+16=32>>32 eggs in total every day.

This leaves her with 16 - 32 = <<16-32=-16>>

Since she sells the remaining eggs at $2 per egg, she makes 16 * $2 = <<16*2=32>>32 at 

In [ ]:
idx = 1
print("QUESTION:", test_questions[idx], "| GOLD:", test_gold[idx], "\n")
for m, i, kind, t in RAW:
    if i == idx and m in ("four_shot", "reflection", "self_consistency"):
        print(f"--- {m} ({kind}) ---\n{t}\n")

QUESTION: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take? | GOLD: 3.0 

--- four_shot (greedy) ---
It takes 2 bolts of blue fiber and half that amount of white fiber, which means it takes 2 * 0.5 = <<2*0.5=1>>1 bolt of white fiber.

Adding the two types of fibers together, it would take 2 + 1 = <<2+1=3>>3 bolts in total.

#### 3

--- self_consistency (sample) ---
The robe takes 2 bolts of blue fiber and half that amount of white fiber, which means it takes 2 + (2 / 2) = <<2+(2/2)=3>>3 bolts of each color.
So in total, it takes 2 + 3 = <<2+3=5>>5 bolts of both blue and white fiber.

--- self_consistency (sample) ---
To find out how many bolts of fabric it takes, we need to calculate both the blue and white fiber separately.

For blue fibers: Let's denote the amount of blue fiber as B. Then the amount of white fiber is half of what it is (i.e., W), which would be B / 2.

Total bolts needed = B (blue) + W (white)

Since there are t

In [ ]:
Counter([5.0, 5.0, 3.0, 3.0]).most_common(1)

[(5.0, 2)]

# Agent Desing

In [ ]:
import ast, operator

OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub,
    ast.Mult: operator.mul, ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) \
            and not isinstance(node.value, bool):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in OPS:
        left, right = _eval(node.left), _eval(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 100:
            raise ValueError("exponent too large")
        return OPS[type(node.op)](left, right)
    if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
        return OPS[type(node.op)](_eval(node.operand))
    raise ValueError("disallowed expression")

def calculator(expr):
    try:
        tree = ast.parse(expr.strip(), mode="eval")
        result = _eval(tree.body)
        if isinstance(result, float) and result == int(result):
            result = int(result)          # 18.0 -> 18 for cleaner observations
        return str(result)
    except ZeroDivisionError:
        return "ERROR: division by zero"
    except Exception as e:
        return f"ERROR: {e}"

In [ ]:
import re

SYSTEM_AGENT = (
    "You solve math word problems using a calculator.\n"
    "Rules:\n"
    "- To calculate, write exactly one line: CALC: <expression>\n"
    "- Use only numbers and + - * / ( ). No commas, no $, no %, no words.\n"
    "- Then stop. You will receive the result as OBSERVATION: <result>.\n"
    "- After at most 3 calculations, write exactly: FINAL: <number>"

)

def demo_to_agent_msgs(d):
    msgs = [{"role": "user", "content": d["question"]}]
    steps = re.findall(r"<<([^=<>]+)=([^<>]+)>>", d["answer"])
    final = d["answer"].split("####")[-1].strip().replace(",", "")
    for expr, res in steps[:3]:
        msgs.append({"role": "assistant", "content": f"CALC: {expr.strip()}"})
        msgs.append({"role": "user", "content": f"OBSERVATION: {res.strip()}"})
    msgs.append({"role": "assistant", "content": f"FINAL: {final}"})
    return msgs

AGENT_DEMOS = [m for d in demos for m in demo_to_agent_msgs(d)]
for m in AGENT_DEMOS:
    print(m["role"].upper(), "|", m["content"][:100])

USER | A deep-sea monster rises from the waters once every hundred years to feast on a ship and sate its hu
ASSISTANT | CALC: 2*2
USER | OBSERVATION: 4
ASSISTANT | CALC: 847/7
USER | OBSERVATION: 121
ASSISTANT | FINAL: 121
USER | Tobias is buying a new pair of shoes that costs $95. He has been saving up his money each month for 
ASSISTANT | CALC: 95+15
USER | OBSERVATION: 110
ASSISTANT | CALC: 3*5
USER | OBSERVATION: 15
ASSISTANT | CALC: 4*15
USER | OBSERVATION: 60
ASSISTANT | FINAL: 5
USER | Randy has 60 mango trees on his farm. He also has 5 less than half as many coconut trees as mango tr
ASSISTANT | CALC: 60/2
USER | OBSERVATION: 30
ASSISTANT | CALC: 30-5
USER | OBSERVATION: 25
ASSISTANT | CALC: 60+25
USER | OBSERVATION: 85
ASSISTANT | FINAL: 85
USER | Jasper will serve charcuterie at his dinner party. He buys 2 pounds of cheddar cheese for $10, a pou
ASSISTANT | CALC: 10/2
USER | OBSERVATION: 5
ASSISTANT | CALC: 10*2
USER | OBSERVATION: 20
ASSISTANT | CALC: 10+5+20
USER | OBSERVAT

In [ ]:
def agent(question, max_calls=3):
    msgs = [{"role": "system", "content": SYSTEM_AGENT}] + AGENT_DEMOS + \
           [{"role": "user", "content": question}]
    trace, n_calc, tin, tout = [], 0, 0, 0
    while True:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        tin += len(tokenizer(text).input_ids)
        out = generate(msgs, max_new_tokens=100)[0]
        tout += len(tokenizer(out).input_ids)

        first = out.strip().split("\n")[0]      # keep only the first line
        trace.append(first)

        m_final = re.match(r"FINAL:\s*\$?(-?[\d,]*\.?\d+)", first)
        if m_final:
            return float(m_final.group(1).replace(",", "")), trace, tin, tout, n_calc


        m_calc = re.match(r"CALC:\s*(.+)", first)
        if m_calc and n_calc < max_calls:
            obs = calculator(m_calc.group(1))
            n_calc += 1
            trace.append(f"OBSERVATION: {obs}")
            msgs.append({"role": "assistant", "content": first})
            msgs.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            if n_calc == max_calls:
                msgs[-1]["content"] += "\nNow write FINAL: <number>."
            continue
        return None, trace, tin, tout, n_calc

In [ ]:
pred, trace, tin, tout, nc = agent(dev_questions[0])
print(pred, dev_gold[0], tin, tout, nc)
print("\n".join(trace))

48.0 72.0 1558 16 1
CALC: 48/2
OBSERVATION: 24
FINAL: 48+24


In [ ]:
def run_agent(questions, golds, verbose=True):
    rows = []
    for i, q in enumerate(questions):
        CUR.update(method="agent", idx=i)
        pred, trace, tin, tout, nc = agent(q)
        correct = pred is not None and abs(pred - golds[i]) < 1e-6
        rows.append(dict(idx=i, pred=pred, gold=golds[i], correct=correct,
                         tin=tin, tout=tout, calc_calls=nc, trace=trace))
        if verbose:
            print(f"[{i}] pred={pred} gold={golds[i]} ok={correct} calc_calls={nc} in={tin} out={tout}")
            print("   " + "\n   ".join(trace))
    return rows

dev_agent = run_agent(dev_questions, dev_gold)
print("ACC:", sum(r["correct"] for r in dev_agent), "/", len(dev_agent))

[0] pred=48.0 gold=72.0 ok=False calc_calls=1 in=1558 out=16
   CALC: 48/2
   OBSERVATION: 24
   FINAL: 48+24
[1] pred=10.0 gold=10.0 ok=True calc_calls=1 in=1546 out=17
   CALC: 12*50/60
   OBSERVATION: 10
   FINAL: 10
[2] pred=95.0 gold=5.0 ok=False calc_calls=2 in=2451 out=29
   CALC: 100/2
   OBSERVATION: 50
   CALC: 50+15+2*(15)
   OBSERVATION: 95
   FINAL: 95
[3] pred=108.0 gold=42.0 ok=False calc_calls=2 in=2449 out=31
   CALC: 120-12=108
   OBSERVATION: ERROR: invalid syntax (<unknown>, line 1)
   CALC: 12*2
   OBSERVATION: 24
   FINAL: 108-24
[4] pred=2184.0 gold=624.0 ok=False calc_calls=1 in=1538 out=19
   CALC: 3*2*7*52
   OBSERVATION: 2184
   FINAL: 2184
[5] pred=100.0 gold=35.0 ok=False calc_calls=1 in=1647 out=43
   CALC: 10*100%/100+(10*100%)+(10*100%/100)*25%
   OBSERVATION: ERROR: invalid syntax (<unknown>, line 1)
   FINAL: 100
[6] pred=48.0 gold=48.0 ok=True calc_calls=1 in=1598 out=17
   CALC: 2*16+2*8
   OBSERVATION: 48
   FINAL: 48
[7] pred=26.0 gold=16.0 ok=Fals

In [ ]:
def agent(question, max_calls=3):
    msgs = [{"role": "system", "content": SYSTEM_AGENT}] + AGENT_DEMOS + \
           [{"role": "user", "content": question}]
    trace, n_calc, tin, tout = [], 0, 0, 0
    while True:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        tin += len(tokenizer(text).input_ids)
        out = generate(msgs, max_new_tokens=100)[0]
        tout += len(tokenizer(out).input_ids)

        first = out.strip().split("\n")[0]
        trace.append(first)

        # FINAL must be a single number and nothing else on the line
        m_final = re.fullmatch(r"FINAL:\s*\$?(-?[\d,]*\.?\d+)\s*", first)
        if m_final:
            return float(m_final.group(1).replace(",", "")), trace, tin, tout, n_calc

        # FINAL with an expression (e.g. "FINAL: 48+24"): give one corrective observation
        if first.startswith("FINAL:") and n_calc < max_calls:
            n_calc += 1
            obs = "ERROR: FINAL must be a single number. Use CALC for the last step."
            trace.append(f"OBSERVATION: {obs}")
            msgs.append({"role": "assistant", "content": first})
            msgs.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            continue

        m_calc = re.match(r"CALC:\s*(.+)", first)
        if m_calc and n_calc < max_calls:
            obs = calculator(m_calc.group(1))
            n_calc += 1
            trace.append(f"OBSERVATION: {obs}")
            msgs.append({"role": "assistant", "content": first})
            msgs.append({"role": "user", "content": f"OBSERVATION: {obs}"})
            if n_calc == max_calls:
                msgs[-1]["content"] += "\nNow write FINAL: <number>."
            continue
        return None, trace, tin, tout, n_calc

In [ ]:

dev_agent = run_agent(dev_questions, dev_gold)
print("ACC:", sum(r["correct"] for r in dev_agent), "/", len(dev_agent))

[0] pred=72.0 gold=72.0 ok=True calc_calls=3 in=3255 out=30
   CALC: 48/2
   OBSERVATION: 24
   FINAL: 48+24
   OBSERVATION: ERROR: FINAL must be a single number. Use CALC for the last step.
   CALC: 48+24
   OBSERVATION: 72
   FINAL: 72
[1] pred=10.0 gold=10.0 ok=True calc_calls=1 in=1546 out=17
   CALC: 12*50/60
   OBSERVATION: 10
   FINAL: 10
[2] pred=95.0 gold=5.0 ok=False calc_calls=2 in=2451 out=29
   CALC: 100/2
   OBSERVATION: 50
   CALC: 50+15+2*(15)
   OBSERVATION: 95
   FINAL: 95
[3] pred=None gold=42.0 ok=False calc_calls=3 in=3336 out=43
   CALC: 120-12=108
   OBSERVATION: ERROR: invalid syntax (<unknown>, line 1)
   CALC: 12*2
   OBSERVATION: 24
   FINAL: 108-24
   OBSERVATION: ERROR: FINAL must be a single number. Use CALC for the last step.
   CALC: 108-24/2
[4] pred=2184.0 gold=624.0 ok=False calc_calls=1 in=1538 out=19
   CALC: 3*2*7*52
   OBSERVATION: 2184
   FINAL: 2184
[5] pred=100.0 gold=35.0 ok=False calc_calls=1 in=1647 out=43
   CALC: 10*100%/100+(10*100%)+(10*

In [ ]:
RAW.clear()
test_agent = run_agent(test_questions, test_gold, verbose=False)
agent_acc  = sum(r["correct"] for r in test_agent)
agent_tin  = sum(r["tin"] for r in test_agent)
agent_tout = sum(r["tout"] for r in test_agent)
agent_calc = sum(r["calc_calls"] for r in test_agent)
agent_gens = len(RAW)
print(agent_acc, "/", len(test_agent), "| input tokens:", agent_tin,
      "| generated tokens:", agent_tout, "| calculator calls:", agent_calc,
      "| model generations:", agent_gens)

8 / 30 | input tokens: 55201 | generated tokens: 756 | calculator calls: 38 | model generations: 68


In [ ]:
def count_input(msgs):
    t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return len(tokenizer(t).input_ids)

fs_in = sum(count_input(four_shot(q)) for q in test_questions)
print("four-shot input tokens:", fs_in)

four-shot input tokens: 26655


In [ ]:
test_agent

[{'idx': 0,
  'pred': 112.0,
  'gold': 18.0,
  'correct': False,
  'tin': 1611,
  'tout': 14,
  'calc_calls': 1,
  'trace': ['CALC: 16*7', 'OBSERVATION: 112', 'FINAL: 112']},
 {'idx': 1,
  'pred': 5.0,
  'gold': 3.0,
  'correct': False,
  'tin': 1534,
  'tout': 15,
  'calc_calls': 1,
  'trace': ['CALC: 2*2+2/2', 'OBSERVATION: 5', 'FINAL: 5']},
 {'idx': 2,
  'pred': 105.0,
  'gold': 70000.0,
  'correct': False,
  'tin': 3431,
  'tout': 72,
  'calc_calls': 3,
  'trace': ['CALC: ($80,000+$50,000) * .15',
   'OBSERVATION: ERROR: invalid syntax (<unknown>, line 1)',
   'CALC: ($80,000+$50,000) * .15',
   'OBSERVATION: ERROR: invalid syntax (<unknown>, line 1)',
   'CALC: ($80,000+$50,000) * .15',
   'OBSERVATION: ERROR: invalid syntax (<unknown>, line 1)',
   'FINAL: 105']},
 {'idx': 3,
  'pred': 540.0,
  'gold': 540.0,
  'correct': True,
  'tin': 1553,
  'tout': 16,
  'calc_calls': 1,
  'trace': ['CALC: 3*3*60', 'OBSERVATION: 540', 'FINAL: 540']},
 {'idx': 4,
  'pred': 90.0,
  'gold': 20.0

In [ ]:
print(test_agent[0]["trace"])
print(sum(r["correct"] for r in test_agent), "/", len(test_agent))

['CALC: 16*7', 'OBSERVATION: 112', 'FINAL: 112']
8 / 30


In [ ]:
print(sum("single number" in " ".join(r["trace"]) for r in test_agent))

0


In [ ]:
   for idx in [18, 25]:
       print(idx, test_questions[idx], "| GOLD:", test_gold[idx])
       print("   " + "\n   ".join(test_agent[idx]["trace"]), "\n")

18 Claire makes a 3 egg omelet every morning for breakfast.  How many dozens of eggs will she eat in 4 weeks? | GOLD: 7.0
   CALC: 3*7*24*4
   OBSERVATION: 2016
   FINAL: 2016 

25 Marie ordered one chicken meal that costs $12, 5 packs of milk that costs $3 each, 4 apples that cost $1.50 each, and some boxes of pizza. Marie paid a total of $50. How many boxes of pizza did Marie order if each box costs $8.50? | GOLD: 2.0
   CALC: 12+3*5+4*1.5+8.5
   OBSERVATION: 41.5
   FINAL: 50 



In [ ]:


   import inspect
   print("fullmatch" in inspect.getsource(agent))
   print(sum("single number" in " ".join(r["trace"]) for r in test_agent))

True
0
